# LangChain을 이용한 댓글 분류 자동화 시스템 구현
이 코드에서는 댓글 데이터를 긍정적 또는 부정적으로 분류하는 간단한 시스템을 구현합니다. 이 시스템은 AI 모델을 사용하여 각 댓글을 분석하고, 긍정적이면 `1`, 부정적이면 `-1`을 출력합니다. 이 코드는 OpenAI API를 사용하며, 주어진 데이터를 기반으로 정확도를 계산합니다.


In [ ]:
!pip install langchain_community cohere openai tiktoken langchain numexpr tabulate llmdantic langchain_openai -q

## API 키 설정

In [1]:
# Load environment variables in a file called .env
# API 키 설정 
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# Check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

API key found and looks good so far!
Google API Key exists and begins AI


In [6]:
! uv pip install openpyxl

Using Python 3.12.12 environment at: C:\Users\jino7\Desktop\llm_engineering-main\.venv
Resolved 2 packages in 74ms
Installed 2 packages in 94ms
 + et-xmlfile==2.0.0
 + openpyxl==3.1.5


## Langchain기반 댓글 감성분석 자동화

In [7]:
import pandas as pd
from openai import ChatCompletion
from langchain_openai import ChatOpenAI  # LangChain 라이브러리에서 ChatOpenAI 가져오기

# 데이터 불러오기
data = pd.read_excel('reviews.xlsx')
print(data.head())



                             Review  Label
0  이 제품 정말 최고예요! 배송도 빠르고 품질도 훌륭합니다.      1
1  완전 실망이에요. 사진이랑 너무 달라요. 환불하고 싶어요.     -1
2    가격 대비 정말 좋은 제품입니다. 재구매 의사 있어요!      1
3           배송이 너무 늦었고 포장도 엉망이었습니다.     -1
4      친절한 고객 서비스 덕분에 문제가 잘 해결됐습니다.      1


In [9]:

# 모델 프롬프트 설정
system_prompt = '''
당신은 리뷰 분석 전문가입니다.
댓글을 읽고, 긍정적이면 1, 부정적이면 -1로 분류하세요.
반드시 정수로만 답하세요

---
'''

# 정확도 계산을 위한 변수 초기화
correct_count = 0
total_count = 0

# OpenAI API 호출 설정
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)

# 리뷰 데이터를 순차적으로 분류
for i, row in data.iterrows():
    print(f'# {i+1}')
    print(f'리뷰: {row["Review"]}')

    # 프롬프트 구성
    prompt = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["Review"]}
    ]

    # API 호출 및 결과 수신
    response = llm.invoke(prompt)
    result = response.content
    print(f'모델 출력: {result} / 실제 라벨: {row["Label"]}')
    # 결과 비교 및 정확도 계산
    if '-1' in result:
        prediction = -1
    else:
        prediction = 1

    if prediction == row["Label"]:
        correct_count += 1

    total_count += 1

    print()

# 최종 정확도 출력
accuracy = correct_count / total_count
print(f'정확도: {accuracy:.2%}')

# 1
리뷰: 이 제품 정말 최고예요! 배송도 빠르고 품질도 훌륭합니다.
모델 출력: 1 / 실제 라벨: 1

# 2
리뷰: 완전 실망이에요. 사진이랑 너무 달라요. 환불하고 싶어요.
모델 출력: -1 / 실제 라벨: -1

# 3
리뷰: 가격 대비 정말 좋은 제품입니다. 재구매 의사 있어요!
모델 출력: 1 / 실제 라벨: 1

# 4
리뷰: 배송이 너무 늦었고 포장도 엉망이었습니다.
모델 출력: -1 / 실제 라벨: -1

# 5
리뷰: 친절한 고객 서비스 덕분에 문제가 잘 해결됐습니다.
모델 출력: 1 / 실제 라벨: 1

# 6
리뷰: 품질이 기대 이하입니다. 돈이 아깝네요.
모델 출력: -1 / 실제 라벨: -1

# 7
리뷰: 디자인이 예쁘고 사용하기 편해요. 만족합니다!
모델 출력: 1 / 실제 라벨: 1

# 8
리뷰: 제품에 불량이 있었는데 교환도 안 해줘서 너무 화가 납니다.
모델 출력: -1 / 실제 라벨: -1

# 9
리뷰: 주변 사람들한테 추천하고 싶을 정도로 마음에 들어요.
모델 출력: 1 / 실제 라벨: 1

# 10
리뷰: 설명서가 너무 불친절하고 제품 성능도 별로예요.
모델 출력: -1 / 실제 라벨: -1

# 11
리뷰: 오래 기다렸는데 받고 나니 정말 만족스러워요!
모델 출력: 1 / 실제 라벨: 1

# 12
리뷰: 포장은 괜찮았는데 내용물이 파손되어 있었어요.
모델 출력: -1 / 실제 라벨: -1

# 13
리뷰: 가성비 최고! 이 가격에 이 품질이면 대박이죠.
모델 출력: 1 / 실제 라벨: 1

# 14
리뷰: 고객센터 연결이 너무 어렵고 불친절했어요.
모델 출력: -1 / 실제 라벨: -1

# 15
리뷰: 생각보다 훨씬 좋아서 깜짝 놀랐어요. 강추합니다.
모델 출력: 1 / 실제 라벨: 1

# 16
리뷰: 색상이 사진과 완전히 달라요. 너무 실망이에요.
모델 출력: -1 / 실제 라벨: -1

# 17
리뷰: 빠른 배송에 꼼꼼한 포장까지 완벽했습니다.
모델 출력: 1 / 실제 라벨: 1

# 18
리뷰: 사용